# Image Preprocessing Utilities for Cricket Object Detection
This notebook provides functions and examples for validating, cropping, resizing, and splitting cricket images for object detection. It is adapted from `preprocess.py`.

In [ ]:
# Imports and constants
import os
import numpy as np
from PIL import Image
from typing import Tuple, Optional
import src.utils
from src.utils import IMAGE_WIDTH, IMAGE_HEIGHT

In [ ]:
# Validate image size
def validate_image_size(image: Image.Image) -> bool:
    width, height = image.size
    return width >= IMAGE_WIDTH and height >= IMAGE_HEIGHT

# Calculate aspect ratio
def calculate_aspect_ratio(width: int, height: int) -> Tuple[int, int]:
    from math import gcd
    divisor = gcd(width, height)
    return width // divisor, height // divisor

# Check if aspect ratio is approximately 4:3
def is_aspect_ratio_4_3(width: int, height: int, tolerance: float = 0.05) -> bool:
    target_ratio = 4 / 3
    actual_ratio = width / height
    return abs(actual_ratio - target_ratio) / target_ratio <= tolerance

In [ ]:
# Resize image
def resize_image(image: Image.Image, target_width: int = IMAGE_WIDTH, target_height: int = IMAGE_HEIGHT) -> Image.Image:
    return image.resize((target_width, target_height), Image.LANCZOS)

# Crop to aspect ratio
def crop_to_aspect_ratio(image: Image.Image, aspect_width: int = 4, aspect_height: int = 3) -> Image.Image:
    width, height = image.size
    target_ratio = aspect_width / aspect_height
    current_ratio = width / height
    if abs(current_ratio - target_ratio) < 0.001:
        return image
    if current_ratio > target_ratio:
        new_width = int(height * target_ratio)
        left = (width - new_width) // 2
        return image.crop((left, 0, left + new_width, height))
    else:
        new_height = int(width / target_ratio)
        top = (height - new_height) // 2
        return image.crop((0, top, width, top + new_height))

In [ ]:
# Preprocess a single image
def preprocess_image(image_path: str, output_path: Optional[str] = None) -> Optional[Image.Image]:
    try:
        image = Image.open(image_path)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        width, height = image.size
        target_ratio = 4 / 3
        current_ratio = width / height
        if current_ratio > target_ratio:
            if height < IMAGE_HEIGHT:
                return None
            potential_width = int(height * target_ratio)
            if potential_width < IMAGE_WIDTH:
                return None
        else:
            if width < IMAGE_WIDTH:
                return None
            potential_height = int(width / target_ratio)
            if potential_height < IMAGE_HEIGHT:
                return None
        image = crop_to_aspect_ratio(image)
        if not validate_image_size(image):
            return None
        image = resize_image(image)
        if output_path:
            output_dir = os.path.dirname(output_path)
            if output_dir:
                os.makedirs(output_dir, exist_ok=True)
            image.save(output_path, quality=95)
        return image
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

In [ ]:
# Preprocess all images in a directory
def preprocess_directory(input_dir: str, output_dir: str, extensions: list = None) -> Tuple[int, int]:
    if extensions is None:
        extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp']
    os.makedirs(output_dir, exist_ok=True)
    processed = 0
    skipped = 0
    for filename in os.listdir(input_dir):
        if not any(filename.lower().endswith(ext) for ext in extensions):
            continue
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        result = preprocess_image(input_path, output_path)
        if result is not None:
            processed += 1
        else:
            skipped += 1
            print(f"Skipped: {filename}")
    return processed, skipped

In [ ]:
# Convert between PIL Image and numpy array
def image_to_numpy(image: Image.Image) -> np.ndarray:
    return np.array(image, dtype=np.uint8)

def numpy_to_image(array: np.ndarray) -> Image.Image:
    return Image.fromarray(array.astype(np.uint8))

In [ ]:
# Split image into grid cells
def extract_grid_cell(image: np.ndarray, cell_index: int) -> np.ndarray:
    from src.utils import get_grid_cell_bounds
    x_start, y_start, x_end, y_end = get_grid_cell_bounds(cell_index)
    return image[y_start:y_end, x_start:x_end]

def split_image_to_grid(image: np.ndarray) -> list:
    try:
        from src.utils import TOTAL_CELLS
    except ImportError:
        from utils import TOTAL_CELLS
    cells = []
    for i in range(1, TOTAL_CELLS + 1):
        cell = extract_grid_cell(image, i)
        cells.append(cell)
    return cells

## Example Usage
Below are example cells to demonstrate how to use the preprocessing functions for a single image and a directory.

In [ ]:
# Example: Preprocess a single image
input_image_path = 'path/to/input.jpg'
output_image_path = 'path/to/output.jpg'
processed_img = preprocess_image(input_image_path, output_image_path)
if processed_img:
    print('Image processed and saved.')
else:
    print('Image did not meet requirements.')

In [ ]:
# Example: Preprocess all images in a directory
input_dir = 'path/to/input_images'
output_dir = 'path/to/output_images'
processed, skipped = preprocess_directory(input_dir, output_dir)
print(f'Processed: {processed}, Skipped: {skipped}')